# Research Agent Improvement Framework - Guided Walkthrough

This notebook shows how the files in `src/deep_agents_foundry/improvement/` connect, by walking the full baseline-vs-candidate pipeline one step at a time.

It runs fully offline using small deterministic fake agents, so every cell returns instantly. No Azure, Foundry, or web-search calls are made here.

> Why the live script felt slow: `scripts/run_improvement_experiment.py` builds a real Research Deep Agent. Each case makes the agent plan, run Foundry web searches, and call the model several times, and the optional judge adds another model call per case. Even a single case can take minutes and consumes quota. This notebook replaces those calls with fakes to teach the mechanics. The final section shows how to run the real experiment.

## How the files connect

```mermaid
flowchart TD
    API["improvement/__init__.py<br/>Public API"]
    DATA["datasets.py<br/>EvaluationCase datasets"]
    DIAG["diagnosis.py<br/>Failure taxonomy + hypotheses"]
    RUN["runner.py<br/>Agent execution + measurements"]
    EVAL["evaluation.py<br/>Existing checks, usage, rubric"]
    MODELS["models.py<br/>RunRecord, comparisons, report"]
    COMP["comparison.py<br/>Deltas + regressions + decision"]
    REPORT["reporting.py<br/>Human-readable report"]
    LOG["experiment_log.py<br/>Compact JSON history"]

    API --> DATA
    API --> DIAG
    API --> RUN
    API --> COMP
    API --> REPORT
    API --> LOG
    DATA --> MODELS
    RUN --> EVAL
    RUN --> MODELS
    COMP --> RUN
    COMP --> MODELS
    REPORT --> MODELS
    LOG --> MODELS
```

The pipeline reads top to bottom:

- `datasets.py` supplies `EvaluationCase` objects built from `models.py`.
- `runner.py` invokes an agent per case and reuses `evaluation.py` for checks, token usage, trajectory features, and optional rubric scoring, recording each run as a `RunRecord`.
- `comparison.py` runs both variants through the runner, then computes deltas, detects regressions, and chooses a conservative decision.
- `reporting.py` renders the structured report.
- `experiment_log.py` explicitly saves a compact JSON summary.
- `diagnosis.py` provides the failure taxonomy and hypothesis model used for bookkeeping.
- `__init__.py` re-exports the supported public surface.

## Setup

Import the public API. Importing `deep_agents_foundry.improvement` does not call Azure; it only wires the modules together.

In [14]:
import json
from pathlib import Path
from types import SimpleNamespace

from deep_agents_foundry.improvement import (
    EvaluationCase,
    ImprovementHypothesis,
    FAILURE_CATEGORIES,
    core_research_dataset,
    dev_cases,
    held_out_cases,
    evaluate_agent_variant,
    compare_agent_variants,
    render_improvement_report,
    save_improvement_experiment,
    load_experiment_record,
    list_experiment_records,
)

print('Imports OK')
print('Failure categories:', FAILURE_CATEGORIES)

Imports OK
Failure categories: ('retrieval', 'reasoning', 'tool_use', 'context', 'memory', 'skill', 'subagent', 'synthesis', 'economics')


## Step 1 - datasets.py: the shared test cases

`datasets.py` provides a compact core dataset with DEV and HELD-OUT subsets. Each `EvaluationCase` carries a prompt, a complexity, and `expected` behavior metadata that the checks interpret.

In [6]:
print('Core cases:', len(core_research_dataset()))
print('DEV cases:', [c.id for c in dev_cases()])
print('HELD-OUT cases:', [c.id for c in held_out_cases()])

example = dev_cases()[0]
print()
print('Example case id:', example.id)
print('  prompt:', example.prompt)
print('  complexity:', example.complexity)
print('  expected:', example.expected)

Core cases: 8
DEV cases: ['simple_definition', 'medium_architecture', 'tool_negative', 'skill_routing', 'memory_preference']
HELD-OUT cases: ['simple_no_overagent', 'complex_comparison', 'subagent_identity']

Example case id: simple_definition
  prompt: What are Microsoft Foundry Hosted Agents?
  complexity: simple
  expected: {'needs_web': True, 'max_searches': 2, 'should_delegate': False}


## Step 2 - diagnosis.py: naming the problem

Before a change, `diagnosis.py` records what we observed and why we expect the change to help. This is bookkeeping for the report, not runtime logic. The `failure_category` must be one of the reusable taxonomy labels.

In [7]:
hypothesis = ImprovementHypothesis(
    symptom='Baseline performs redundant web searches on simple questions',
    failure_category='tool_use',
    hypothesis='A tighter search policy reduces searches without hurting quality',
    proposed_change='Enable the technology-research skill and stop-search guidance',
    primary_metric='web_searches',
    guardrail_metric='quality',
)
print('Symptom:         ', hypothesis.symptom)
print('Failure category:', hypothesis.failure_category)
print('Primary metric:  ', hypothesis.primary_metric)
print('Guardrail metric:', hypothesis.guardrail_metric)

Symptom:          Baseline performs redundant web searches on simple questions
Failure category: tool_use
Primary metric:   web_searches
Guardrail metric: quality


## Step 3 - fake agents and a fake judge

The framework never builds the agent for you: you pass already-built agents. Here we build two deterministic fakes so the pipeline runs offline.

A run result mimics a LangGraph/Deep Agent response: a dict with a `messages` list. The runner reads `content_blocks` for server-side web searches, `usage_metadata` for tokens, and the last message content for the final answer.

- Baseline: 3 web searches, higher token use.
- Candidate: 2 web searches, lower token use, and a slightly stronger answer.

The fake judge simulates a higher score when the answer is more complete, standing in for a real rubric model.

In [8]:
def msg(content=None, tool_calls=None, content_blocks=None, usage_metadata=None):
    return SimpleNamespace(
        content=content,
        tool_calls=tool_calls,
        content_blocks=content_blocks,
        usage_metadata=usage_metadata,
    )


class FakeAgent:
    'Async agent stand-in that returns a scripted result and records its calls.'

    def __init__(self, script):
        self._script = script
        self.calls = []

    async def ainvoke(self, payload, config=None, context=None):
        self.calls.append({'config': config, 'context': context})
        return self._script(payload, config, context)


class FakeJudge:
    'Rubric-model stand-in: a fuller answer earns a higher overall score.'

    def invoke(self, prompt):
        score = 4.3 if 'comprehensively' in prompt else 3.7
        return SimpleNamespace(content=json.dumps({'weighted_overall_score': score}))


def baseline_script(payload, config, context):
    return {'messages': [
        msg(content_blocks=[{'type': 'server_tool_call'}]),
        msg(content_blocks=[{'type': 'server_tool_call'}]),
        msg(
            content_blocks=[{'type': 'server_tool_call'}],
            usage_metadata={'input_tokens': 1400, 'output_tokens': 300, 'total_tokens': 1700},
        ),
        msg(
            content='Foundry Hosted Agents are managed agents. Source: https://learn.microsoft.com',
            usage_metadata={'input_tokens': 500, 'output_tokens': 180, 'total_tokens': 680},
        ),
    ]}


def candidate_script(payload, config, context):
    return {'messages': [
        msg(content_blocks=[{'type': 'server_tool_call'}]),
        msg(
            content_blocks=[{'type': 'server_tool_call'}],
            usage_metadata={'input_tokens': 1100, 'output_tokens': 260, 'total_tokens': 1360},
        ),
        msg(
            content='Foundry Hosted Agents are managed, comprehensively described agents. Source: https://learn.microsoft.com',
            usage_metadata={'input_tokens': 300, 'output_tokens': 120, 'total_tokens': 420},
        ),
    ]}


baseline_agent = FakeAgent(baseline_script)
candidate_agent = FakeAgent(candidate_script)
judge = FakeJudge()

demo_cases = [
    EvaluationCase(
        id='definition',
        prompt='What are Microsoft Foundry Hosted Agents?',
        complexity='simple',
        expected={'needs_web': True, 'max_searches': 4},
    ),
    EvaluationCase(
        id='architecture',
        prompt='Explain the architecture and trade-offs of Hosted Agents.',
        complexity='medium',
        expected={'needs_web': True, 'max_searches': 6},
    ),
]
print('Fake agents and demo dataset ready.')

Fake agents and demo dataset ready.


## Step 4 - runner.py + evaluation.py: run one variant

`evaluate_agent_variant` runs an agent over the dataset with an isolated thread per case. For each case it reuses `evaluation.py` to collect deterministic checks, token usage, trajectory features, and (when a judge is supplied) a rubric score, packaging everything into a `RunRecord`.

Metrics the runtime does not expose stay `None` rather than a misleading zero.

In [9]:
baseline_result = await evaluate_agent_variant(
    baseline_agent, demo_cases, variant_label='baseline', judge_model=judge
)

record = baseline_result.records[0]
print('variant:', baseline_result.label)
print('case id:', record.case_id)
print('thread id:', record.thread_id)
print('final text:', record.final_text)
print('latency seconds:', round(record.latency_seconds, 5))
print('metrics:', record.metrics)
print('deterministic checks:', record.deterministic)
print('boolean checks:', record.checks)
print('rubric quality:', record.quality)

variant: baseline
case id: definition
thread id: improve-baseline-definition-6b1624b0
final text: Foundry Hosted Agents are managed agents. Source: https://learn.microsoft.com
latency seconds: 2e-05
metrics: {'input_tokens': 1900, 'output_tokens': 480, 'total_tokens': 2380, 'model_calls': 2, 'web_searches': 3, 'tool_calls': 0, 'subagent_calls': 0}
deterministic checks: {'completed': True, 'searched_when_expected': True, 'citations_present': True, 'elapsed_seconds': 1.880002673715353e-05, 'server_tool_calls': 3}
boolean checks: {'completed': True, 'searched_when_expected': True, 'citations_present': True, 'answer_non_empty': True, 'within_search_budget': True}
rubric quality: 3.7


## Step 5 - comparison.py: baseline vs candidate

`compare_agent_variants` runs both agents through the same runner, then computes per-case and aggregate deltas, detects regressions, and produces a conservative `KEEP` / `REVERT` / `INVESTIGATE` decision. It never collapses everything into one opaque score.

In [10]:
report = await compare_agent_variants(
    baseline_agent,
    candidate_agent,
    dataset=demo_cases,
    judge_model=judge,
    baseline_label='baseline',
    candidate_label='candidate',
)

print('Recommendation:', report.recommendation)
print('Evidence:', report.evidence)
print('Reason:', report.recommendation_reason)
print('Overall quality delta:', report.quality_delta)
print()
print('Per-case deltas (candidate - baseline):')
for case in report.case_comparisons:
    print(
        f'  {case.case_id}: quality {case.quality_delta:+.2f}'
        f' | web_searches {case.deltas.get("web_searches")}'
        f' | total_tokens {case.deltas.get("total_tokens")}'
    )

Recommendation: KEEP
Evidence: clear
Reason: Overall rubric quality improved +0.60 with no critical regressions.
Overall quality delta: 0.5999999999999996

Per-case deltas (candidate - baseline):
  definition: quality +0.60 | web_searches -1.0 | total_tokens -600.0
  architecture: quality +0.60 | web_searches -1.0 | total_tokens -600.0


## Step 6 - reporting.py: the human-readable report

`render_improvement_report` turns the structured `ComparisonReport` into readable text with QUALITY, TRAJECTORY, PERFORMANCE, REGRESSIONS, OBSERVATIONS, and RECOMMENDATION sections. Unavailable metrics render as `n/a`.

In [11]:
print(render_improvement_report(report))

Research Agent Improvement Report

candidate vs baseline
Cases compared: 2

QUALITY
- Overall rubric quality delta: +0.60
- Cases improved: 2, regressed: 0, total: 2

TRAJECTORY
- Avg web searches: 3.00 -> 2.00 (delta -1.00 (-33.3%))
- Avg model calls: 2.00 -> 2.00 (delta +0.00 (+0.0%))
- Avg tool calls: 0.00 -> 0.00 (delta +0.00)
- Avg subagent calls: 0.00 -> 0.00 (delta +0.00)

PERFORMANCE
- Avg total tokens: 2380.00 -> 1780.00 (delta -600.00 (-25.2%))
- Avg latency (s): 0.00 -> 0.00 (delta -0.00 (-31.6%))

REGRESSIONS
- None detected

OBSERVATIONS
✓ Overall rubric quality delta: +0.60
✓ Average web searches decreased.
✓ Average tokens changed -25.2%.
✓ Improved cases: definition, architecture.

RECOMMENDATION
KEEP

Evidence: clear
Reason:
Overall rubric quality improved +0.60 with no critical regressions.


## Step 7 - experiment_log.py: save and reload history

Saving is explicit: the comparison never writes files on its own. `save_improvement_experiment` converts the report into a compact `ExperimentRecord` and writes **one JSON file per experiment**. Raw answers and traces are intentionally excluded.

Here we save inside the package at `improvement/artifacts/experiments/`, so reports and logs live next to the code you reference. The path is anchored to the package location, so it does not depend on the working directory.

Why one file per experiment instead of a single appended file:

- Each run is an immutable record, so history cannot be accidentally overwritten.
- There is no read-modify-write step, which avoids corrupting the file on a crash or a concurrent run.
- A single experiment can be inspected, shared, or deleted on its own.
- Trend analysis is still easy: `list_experiment_records(directory=...)` reads every file back, and you build a DataFrame from that list (shown below).

In [15]:
import deep_agents_foundry.improvement as improvement_pkg

# Save alongside the package so reports live inside improvement/artifacts/.
artifacts_dir = Path(improvement_pkg.__file__).parent / 'artifacts' / 'experiments'

saved_path = save_improvement_experiment(
    report,
    change_id='demo-skill-change-v1',
    observed_failure='Redundant web searches on simple questions',
    hypothesis='The candidate improves quality while searching less',
    change_description='Synthetic demo comparison using deterministic fake agents',
    notes='Notebook demonstration only; not a real experiment.',
    directory=artifacts_dir,
)
print('Saved to:', saved_path)
print()

loaded = load_experiment_record(saved_path)
print('Reloaded decision:', loaded.decision)
print('Reloaded quality delta:', loaded.quality_delta)
print('Reloaded deltas:', loaded.deltas)
print()

print('All records in the directory:')
for rec in list_experiment_records(directory=artifacts_dir):
    print(' ', rec.timestamp, rec.change_id, rec.decision)

Saved to: C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\improvement\artifacts\experiments\2026-09-13T172850Z_demo-skill-change-v1.json

Reloaded decision: KEEP
Reloaded quality delta: 0.5999999999999996
Reloaded deltas: {'total_tokens': -600.0, 'latency_seconds': -3.5500270314514637e-06, 'web_searches': -1.0, 'model_calls': 0.0, 'tool_calls': 0.0, 'subagent_calls': 0.0}

All records in the directory:
  2026-09-13T17:28:50.665331+00:00 demo-skill-change-v1 KEEP


In [16]:
print(saved_path.read_text(encoding='utf-8'))

{
  "change_id": "demo-skill-change-v1",
  "baseline_label": "baseline",
  "candidate_label": "candidate",
  "decision": "KEEP",
  "observed_failure": "Redundant web searches on simple questions",
  "hypothesis": "The candidate improves quality while searching less",
  "change_description": "Synthetic demo comparison using deterministic fake agents",
  "quality_delta": 0.5999999999999996,
  "deltas": {
    "total_tokens": -600.0,
    "latency_seconds": -3.5500270314514637e-06,
    "web_searches": -1.0,
    "model_calls": 0.0,
    "tool_calls": 0.0,
    "subagent_calls": 0.0
  },
  "regressions": [],
  "evidence": "clear",
  "recommendation_reason": "Overall rubric quality improved +0.60 with no critical regressions.",
  "notes": "Notebook demonstration only; not a real experiment.",
  "timestamp": "2026-09-13T17:28:50.665331+00:00"
}


### Trend analysis reads the whole directory

Separate files do not make trend analysis harder. Load every record into a DataFrame in one step, then chart or aggregate it exactly as the trend-analysis notebook does with the synthetic history.

In [17]:
import pandas as pd

# Separate files are still easy to analyze: load them all in one step.
history = list_experiment_records(directory=artifacts_dir)
history_df = pd.DataFrame(
    {
        'timestamp': record.timestamp,
        'change_id': record.change_id,
        'decision': record.decision,
        'quality_delta': record.quality_delta,
        **{f'delta_{name}': value for name, value in record.deltas.items()},
    }
    for record in history
)
history_df

,timestamp,change_id,decision,quality_delta,delta_total_tokens,delta_latency_seconds,delta_web_searches,delta_model_calls,delta_tool_calls,delta_subagent_calls
0,2026-09-13T17:28:50.665331+00:00,demo-skill-change-v1,KEEP,0.6,-600.0,-0.000004,-1.0,0.0,0.0,0.0


## Step 8 - running the real experiment

Everything above used fakes. To measure an actual change, build two real agents and pass them to the same pipeline. This makes live Foundry and model calls, so it is slow and consumes quota.

```python
from deep_agents_foundry import build_model, build_research_agent

baseline_agent = build_research_agent()
candidate_agent = build_research_agent(skills=['.'])  # Skills enabled
judge_model = build_model()

report = await compare_agent_variants(
    baseline_agent,
    candidate_agent,
    dataset=dev_cases(),
    judge_model=judge_model,
    baseline_label='skills-off',
    candidate_label='skills-on',
)
print(render_improvement_report(report))
```

Or use the CLI script, which does the same thing:

```powershell
uv run python scripts/run_improvement_experiment.py --suite targeted-dev --change-id technology-skill-v2
```

Why the live run is slow, and how to shrink it:

- Each case runs a real Deep Agent that plans, performs Foundry web searches, and calls the model several times.
- With a judge, add one more model call per case.
- Multiply by the number of cases and by two variants.

For a fast smoke check, run one case and skip the judge:

```powershell
uv run python scripts/run_improvement_experiment.py --case skill_routing --no-judge --change-id smoke-check
```

Even then, a real agent invocation can take from tens of seconds to a few minutes depending on how many searches it performs.

## Recap

`datasets.py` defines the cases, `runner.py` executes an agent and reuses `evaluation.py` to measure it, `comparison.py` turns two variants into deltas and a decision, `reporting.py` makes it readable, and `experiment_log.py` saves a compact record. `diagnosis.py` and `models.py` supply the shared vocabulary and data structures, and `__init__.py` exposes the public API. The framework measures and explains a change; it never builds the agent or decides to ship for you.